In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shopee-product-matching/sample_submission.csv
/kaggle/input/competitions/shopee-product-matching/train.csv
/kaggle/input/competitions/shopee-product-matching/test.csv


KeyboardInterrupt: 

In [6]:
#PART B
import re
import unicodedata

# -----------------------------------
# 1. Load dataset
# -----------------------------------

df = pd.read_csv("/kaggle/input/competitions/shopee-product-matching/train.csv")

print("Dataset shape:", df.shape)
print(train_df.columns.tolist())


# -----------------------------------
# 2. Inspect missing titles
# -----------------------------------

print("Missing titles:", df["title"].isna().sum())


# -----------------------------------
# 3. Keep original title
# -----------------------------------

df["title_original"] = df["title"]


# -----------------------------------
# 4. Preprocessing function
# -----------------------------------

def preprocess_title(text):

    # Missing value
    if pd.isna(text):
        return ""

    # Convert to string
    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text


# -----------------------------------
# 5. Apply preprocessing
# -----------------------------------

df["title_clean"] = df["title"].apply(preprocess_title)


# -----------------------------------
# 6. Inspect results
# -----------------------------------

print(
    df[
        ["title_original", "title_clean"]
    ].head(20).to_string(index=False)
)


# -----------------------------------
# 7. Check empty titles
# -----------------------------------

empty_count = (
    df["title_clean"]
    .str.strip()
    .eq("")
    .sum()
)

print("Empty titles after preprocessing:", empty_count)

Dataset shape: (34250, 5)
['posting_id', 'image', 'image_phash', 'title', 'label_group']
Missing titles: 0
                                                                                    title_original                                                                                        title_clean
                                                                         Paper Bag Victoria Secret                                                                          paper bag victoria secret
                                      Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DOUBLE FOAM TAPE                                       double tape 3m vhb 12 mm x 4,5 m original / double foam tape
                                                       Maling TTS Canned Pork Luncheon Meat 397 gr                                                        maling tts canned pork luncheon meat 397 gr
Daster Batik Lengan pendek - Motif Acak / Campur - Leher Kancing (DPT001-00) Batik karakter Alhadi da

In [ ]:
# ============================================================
# SHOPEE PRODUCT MATCHING — TF-IDF PIPELINE
# From title_clean → Train/Validation → TF-IDF → Pair Creation
# → Cosine Similarity → Threshold Sweep → Model Comparison
# ============================================================



from sklearn.model_selection import GroupShuffleSplit

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)


# ============================================================
# 4. REMOVE EMPTY TITLES
# ============================================================

before = len(df)

df = df[
    df["title_clean"].str.strip() != ""
].reset_index(drop=True)

after = len(df)

print("\nRemoved empty titles:", before - after)
print("Remaining listings:", after)


# ============================================================
# 5. GROUP-BASED TRAIN / VALIDATION SPLIT
#
# IMPORTANT:
# label_group represents the underlying product.
#
# We split entire groups rather than individual rows.
# This prevents the same product group appearing in both
# train and validation.
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(
        df,
        groups=df["label_group"]
    )
)

train_df = df.iloc[
    train_idx
].reset_index(drop=True)

val_df = df.iloc[
    val_idx
].reset_index(drop=True)


print("\n" + "=" * 60)
print("TRAIN / VALIDATION SPLIT")
print("=" * 60)

print("Train listings:", len(train_df))
print("Validation listings:", len(val_df))

print(
    "Train product groups:",
    train_df["label_group"].nunique()
)

print(
    "Validation product groups:",
    val_df["label_group"].nunique()
)


# ============================================================
# 6. VERIFY NO GROUP LEAKAGE
# ============================================================

train_groups = set(
    train_df["label_group"]
)

val_groups = set(
    val_df["label_group"]
)

overlap = train_groups.intersection(
    val_groups
)

print(
    "Overlapping product groups:",
    len(overlap)
)

assert len(overlap) == 0, \
    "ERROR: Product-group leakage detected!"


# ============================================================
# 7. GENERATE POSITIVE PAIRS
#
# Same label_group = same underlying product.
#
# Example:
#
# A → group 100
# B → group 100
#
# (A,B) = positive pair
# ============================================================

def generate_positive_pairs(
    data,
    max_pairs_per_group=20,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    pairs = []

    grouped = data.groupby(
        "label_group"
    )

    for group_id, group in grouped:

        indices = group.index.tolist()

        # Need at least 2 listings
        if len(indices) < 2:
            continue

        possible_pairs = []

        for i in range(len(indices)):

            for j in range(
                i + 1,
                len(indices)
            ):

                possible_pairs.append(
                    (
                        indices[i],
                        indices[j]
                    )
                )

        # Prevent very large groups from
        # generating enormous numbers of pairs
        if len(possible_pairs) > max_pairs_per_group:

            selected_indices = rng.choice(
                len(possible_pairs),
                size=max_pairs_per_group,
                replace=False
            )

            possible_pairs = [
                possible_pairs[i]
                for i in selected_indices
            ]

        for a, b in possible_pairs:

            pairs.append(
                (
                    a,
                    b,
                    1
                )
            )

    return pairs


positive_pairs = generate_positive_pairs(
    val_df,
    max_pairs_per_group=20,
    random_state=42
)

print("\nPositive pairs:", len(positive_pairs))


# ============================================================
# 8. GENERATE NEGATIVE PAIRS
#
# Different label_group = different products.
# ============================================================

def generate_negative_pairs(
    data,
    num_pairs,
    random_state=42
):

    rng = np.random.default_rng(
        random_state
    )

    indices = np.arange(
        len(data)
    )

    groups = data[
        "label_group"
    ].values

    pairs = set()

    while len(pairs) < num_pairs:

        a, b = rng.choice(
            indices,
            size=2,
            replace=False
        )

        # Must belong to different products
        if groups[a] == groups[b]:
            continue

        pair = (
            min(a, b),
            max(a, b)
        )

        pairs.add(pair)

    return [
        (
            a,
            b,
            0
        )
        for a, b in pairs
    ]


negative_pairs = generate_negative_pairs(
    val_df,
    num_pairs=len(positive_pairs),
    random_state=42
)

print("Negative pairs:", len(negative_pairs))


# ============================================================
# 9. COMBINE POSITIVE + NEGATIVE PAIRS
# ============================================================

pairs = (
    positive_pairs +
    negative_pairs
)

rng = np.random.default_rng(42)

rng.shuffle(pairs)

print(
    "Total evaluation pairs:",
    len(pairs)
)


# ============================================================
# 10. EXTRACT PAIR INFORMATION
# ============================================================

pair_a = np.array([
    pair[0]
    for pair in pairs
])

pair_b = np.array([
    pair[1]
    for pair in pairs
])

y_true = np.array([
    pair[2]
    for pair in pairs
])


# ============================================================
# 11. TF-IDF MODEL FUNCTION
# ============================================================

def build_tfidf(
    train_text,
    val_text,
    analyzer,
    ngram_range,
    min_df=2,
    max_df=0.95,
    max_features=None
):

    vectorizer = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        max_features=max_features,
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(
        train_text
    )

    X_val = vectorizer.transform(
        val_text
    )

    return (
        vectorizer,
        X_train,
        X_val
    )


# ============================================================
# 12. EXPERIMENT 1
# WORD UNIGRAM TF-IDF
# ============================================================

print("\n" + "=" * 60)
print("EXPERIMENT 1 — WORD UNIGRAM TF-IDF")
print("=" * 60)

word_uni_vectorizer, X_train_word_uni, X_val_word_uni = \
    build_tfidf(
        train_df["title_clean"],
        val_df["title_clean"],
        analyzer="word",
        ngram_range=(1, 1)
    )

print(
    "Train matrix:",
    X_train_word_uni.shape
)

print(
    "Validation matrix:",
    X_val_word_uni.shape
)

print(
    "Vocabulary:",
    len(word_uni_vectorizer.vocabulary_)
)


# ============================================================
# 13. EXPERIMENT 2
# WORD UNIGRAM + BIGRAM TF-IDF
# ============================================================

print("\n" + "=" * 60)
print("EXPERIMENT 2 — WORD 1-2 GRAM TF-IDF")
print("=" * 60)

word_bigram_vectorizer, X_train_word_bigram, X_val_word_bigram = \
    build_tfidf(
        train_df["title_clean"],
        val_df["title_clean"],
        analyzer="word",
        ngram_range=(1, 2)
    )

print(
    "Train matrix:",
    X_train_word_bigram.shape
)

print(
    "Validation matrix:",
    X_val_word_bigram.shape
)

print(
    "Vocabulary:",
    len(word_bigram_vectorizer.vocabulary_)
)


# ============================================================
# 14. EXPERIMENT 3
# CHARACTER 3-5 GRAM TF-IDF
# ============================================================

print("\n" + "=" * 60)
print("EXPERIMENT 3 — CHARACTER 3-5 GRAM TF-IDF")
print("=" * 60)

char_vectorizer, X_train_char, X_val_char = \
    build_tfidf(
        train_df["title_clean"],
        val_df["title_clean"],
        analyzer="char",
        ngram_range=(3, 5),
        max_features=200000
    )

print(
    "Train matrix:",
    X_train_char.shape
)

print(
    "Validation matrix:",
    X_val_char.shape
)

print(
    "Vocabulary:",
    len(char_vectorizer.vocabulary_)
)


# ============================================================
# 15. PAIRWISE COSINE SIMILARITY
# ============================================================

def pairwise_cosine_scores(
    X,
    pairs
):

    scores = []

    for a, b, _ in pairs:

        similarity = cosine_similarity(
            X[a],
            X[b]
        )[0, 0]

        scores.append(
            similarity
        )

    return np.array(scores)


# ============================================================
# 16. CALCULATE SCORES FOR ALL THREE MODELS
# ============================================================

print("\nCalculating Word Unigram scores...")

word_uni_scores = pairwise_cosine_scores(
    X_val_word_uni,
    pairs
)


print("Calculating Word 1-2 gram scores...")

word_bigram_scores = pairwise_cosine_scores(
    X_val_word_bigram,
    pairs
)


print("Calculating Character 3-5 gram scores...")

char_scores = pairwise_cosine_scores(
    X_val_char,
    pairs
)


# ============================================================
# 17. CREATE EVALUATION DATAFRAME
# ============================================================

evaluation_df = pd.DataFrame({

    "idx_a": pair_a,

    "idx_b": pair_b,

    "y_true": y_true,

    "word_unigram_score":
        word_uni_scores,

    "word_bigram_score":
        word_bigram_scores,

    "char_score":
        char_scores
})


# Add titles for inspection later
evaluation_df["title_a"] = (
    evaluation_df["idx_a"]
    .map(
        val_df["title_clean"]
    )
)

evaluation_df["title_b"] = (
    evaluation_df["idx_b"]
    .map(
        val_df["title_clean"]
    )
)


print("\nEvaluation dataframe:")
print(
    evaluation_df.head()
)


# ============================================================
# 18. THRESHOLD PREDICTION
# ============================================================

def predict_from_threshold(
    scores,
    threshold
):

    return (
        scores >= threshold
    ).astype(int)


# ============================================================
# 19. EVALUATION FUNCTION
# ============================================================

def evaluate_threshold(
    y_true,
    scores,
    threshold
):

    y_pred = predict_from_threshold(
        scores,
        threshold
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    }


# ============================================================
# 20. THRESHOLD RANGE
# ============================================================

thresholds = np.arange(
    0.10,
    0.96,
    0.05
)


# ============================================================
# 21. FUNCTION TO RUN THRESHOLD SWEEP
# ============================================================

def threshold_sweep(
    y_true,
    scores,
    thresholds
):

    results = []

    for threshold in thresholds:

        result = evaluate_threshold(
            y_true,
            scores,
            threshold
        )

        results.append(
            result
        )

    return pd.DataFrame(
        results
    )


# ============================================================
# 22. WORD UNIGRAM EVALUATION
# ============================================================

word_uni_results = threshold_sweep(
    y_true,
    word_uni_scores,
    thresholds
)

best_word_uni = word_uni_results.loc[
    word_uni_results["f1"].idxmax()
]

print("\nBest Word Unigram:")
print(best_word_uni)


# ============================================================
# 23. WORD BIGRAM EVALUATION
# ============================================================

word_bigram_results = threshold_sweep(
    y_true,
    word_bigram_scores,
    thresholds
)

best_word_bigram = word_bigram_results.loc[
    word_bigram_results["f1"].idxmax()
]

print("\nBest Word 1-2 Gram:")
print(best_word_bigram)


# ============================================================
# 24. CHARACTER TF-IDF EVALUATION
# ============================================================

char_results = threshold_sweep(
    y_true,
    char_scores,
    thresholds
)

best_char = char_results.loc[
    char_results["f1"].idxmax()
]

print("\nBest Character TF-IDF:")
print(best_char)


# ============================================================
# 25. HYBRID WORD + CHARACTER MODEL
#
# S_hybrid =
#
# alpha * S_word
# +
# (1-alpha) * S_char
# ============================================================

alphas = np.arange(
    0.0,
    1.01,
    0.1
)

hybrid_results = []


for alpha in alphas:

    hybrid_scores = (
        alpha * word_bigram_scores
        +
        (1 - alpha) * char_scores
    )

    for threshold in thresholds:

        result = evaluate_threshold(
            y_true,
            hybrid_scores,
            threshold
        )

        result["alpha"] = alpha

        hybrid_results.append(
            result
        )


hybrid_results = pd.DataFrame(
    hybrid_results
)


# ============================================================
# 26. BEST HYBRID CONFIGURATION
# ============================================================

best_hybrid = hybrid_results.loc[
    hybrid_results["f1"].idxmax()
]

print("\nBest Hybrid Model:")
print(best_hybrid)


# ============================================================
# 27. FINAL MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame([

    {
        "model":
            "Word Unigram TF-IDF",

        "threshold":
            best_word_uni["threshold"],

        "precision":
            best_word_uni["precision"],

        "recall":
            best_word_uni["recall"],

        "f1":
            best_word_uni["f1"],

        "accuracy":
            best_word_uni["accuracy"]
    },

    {
        "model":
            "Word 1-2 Gram TF-IDF",

        "threshold":
            best_word_bigram["threshold"],

        "precision":
            best_word_bigram["precision"],

        "recall":
            best_word_bigram["recall"],

        "f1":
            best_word_bigram["f1"],

        "accuracy":
            best_word_bigram["accuracy"]
    },

    {
        "model":
            "Character 3-5 Gram TF-IDF",

        "threshold":
            best_char["threshold"],

        "precision":
            best_char["precision"],

        "recall":
            best_char["recall"],

        "f1":
            best_char["f1"],

        "accuracy":
            best_char["accuracy"]
    },

    {
        "model":
            "Hybrid Word + Character TF-IDF",

        "threshold":
            best_hybrid["threshold"],

        "precision":
            best_hybrid["precision"],

        "recall":
            best_hybrid["recall"],

        "f1":
            best_hybrid["f1"],

        "accuracy":
            best_hybrid["accuracy"]
    }

])


print("\n" + "=" * 60)
print("FINAL TF-IDF COMPARISON")
print("=" * 60)

print(
    comparison
    .sort_values(
        "f1",
        ascending=False
    )
    .to_string(index=False)
)


# ============================================================
# 28. SAVE RESULTS
# ============================================================

comparison.to_csv(
    "tfidf_model_comparison.csv",
    index=False
)

word_uni_results.to_csv(
    "tfidf_word_unigram_thresholds.csv",
    index=False
)

word_bigram_results.to_csv(
    "tfidf_word_bigram_thresholds.csv",
    index=False
)

char_results.to_csv(
    "tfidf_character_thresholds.csv",
    index=False
)

hybrid_results.to_csv(
    "tfidf_hybrid_thresholds.csv",
    index=False
)

evaluation_df.to_csv(
    "tfidf_pair_evaluation.csv",
    index=False
)


print("\nResults saved.")


# ============================================================
# 29. SELECT FINAL TF-IDF CONFIGURATION
# ============================================================

best_model_row = comparison.loc[
    comparison["f1"].idxmax()
]

print("\n" + "=" * 60)
print("SELECTED TF-IDF CONFIGURATION")
print("=" * 60)

print(
    "Model:",
    best_model_row["model"]
)

print(
    "Threshold:",
    best_model_row["threshold"]
)

print(
    "Precision:",
    best_model_row["precision"]
)

print(
    "Recall:",
    best_model_row["recall"]
)

print(
    "F1:",
    best_model_row["f1"]
)

print(
    "Accuracy:",
    best_model_row["accuracy"]
)